In [2]:
%pip install -q transformers
%pip install -q torch
%pip install -q datasets
%pip install -q sentencepiece  


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch import nn
from torch.utils.data import DataLoader
import os
from tqdm import tqdm

In [4]:

from datasets import load_dataset

In [6]:
#Import libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch import nn
import os

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


prune only 20% of the least important neurons based on their L1 norm, aiming to balance size reduction with minimal accuracy loss.

You can adjust this percentage and increase it to 30% or even 50%, depending on whether you plan to follow up with a post-pruning fine-tuning process.


In [7]:
prune_percent = 0.2  # Prune 20% of neurons
model_name = 'distilgpt2'

In [8]:

# Support function to check the size reduction.
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

On the other hand, pruning, allows for targeted removal of less important weights or neurons, which can lead to a more efficient reduction in model size while better preserving accuracy. By selecting the weights to eliminate based on their importance scores, pruning provides more control over the model's structure, often making it a more effective approach when aiming for both model compression and high performance.

The effectiveness of removing specific parts of a model could be debated, but recent studies, such as the one conducted by NVIDIA: How to Prune and Distill Llama-3.1 8B to an NVIDIA Llama-3.1-Minitron 4B Model, concluding that pruning, combined with fine-tuning techniques applied after pruning, can produce models that are not only more efficient but also more effective in specific domains.

Also you can combine both techniques and quantize a model that has been previousluy pruned.

This notebook focuses on structured width pruning, where entire neurons are eliminated based on their low importance scores, which are computed using the L1 norm. The assumption is that neurons with lower L1 norm values contribute less to the overall output of the model, allowing for safe removal to enhance efficiency without drastically impacting accuracy.

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
 
 
 

from huggingface_hub import login
login()

In [15]:
# Download the model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
def get_ouput(prompt, model=model, tokenizer=tokenizer):
  inputs = tokenizer(prompt, return_tensors='pt').to(device)
  outputs = model.generate(inputs['input_ids'],
                           attention_mask=inputs['attention_mask'],
                           max_length=10,
                           num_return_sequences=1)
  generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return generated



In the pruning process for the notebook, I only targeted the mlp layers because they typically contribute the most to the model's size and pruning them doesn’t affect the attention mechanism, which is critical for capturing relationships within the input data. These layers also tend to contain more redundancies, and reducing neurons in them generally doesn't significantly impact the model's output—though this always depends on the neuron selection process for elimination.